# Hetionet EDA 全程教程
## Tutorial for Understanding the Full EDA Codebase

> **适用读者 Target Audience**：有 Python 基础（知道变量/循环/函数），但没接触过图数据分析生态的学习者。  
> Learners who know Python basics but haven't worked with graph data analysis libraries.

### 章节目录 Table of Contents

| 章节 | 主题 | 对应 EDA 文件 |
|---|---|---|
| **Ch.0** | Python 基础语法补充 | 全部 |
| **Ch.1** | 数据结构进阶（推导式、defaultdict、Counter） | `utils.py` |
| **Ch.2** | 文件读取与路径处理（pathlib / bz2 / json） | `utils.py` |
| **Ch.3** | Pandas 数据分析 | `01_overview.ipynb` |
| **Ch.4** | Matplotlib / Seaborn 可视化 | 全部 notebook |
| **Ch.5** | NetworkX 图论 | `02_structure.ipynb` / `03_compound_disease.ipynb` |
| **Ch.6** | NumPy / SciPy 科学计算与幂律拟合 | `02_structure.ipynb` |

---
**约定 Conventions**
- 🇨🇳 中文解释原理，🇬🇧 English for terminology and code comments
- `# ← 注释` 指向关键语法点
- 每章末尾配有 **练习题 + 参考答案 Exercises & Answers**
- 所有代码可在 `hetionet-eda` kernel 下直接运行

---
# Ch.0 Python 基础语法补充
## Python Syntax Refresher

本章快速过一遍 EDA 代码中出现的 Python 基础语法，已熟悉可跳过。  
Quick refresher on Python basics used throughout the EDA. Skip if already familiar.

### 0.1 变量与基本类型 Variables & Basic Types

Python 是**动态类型**语言——变量无需声明类型，赋值即创建。  
Python is **dynamically typed**: variables are created by assignment, no declaration needed.

In [ ]:
# 基本类型 Basic types
name        = 'Hetionet'    # str  字符串
n_nodes     = 47031         # int  整数
density     = 0.0036        # float 浮点数
is_directed = True          # bool 布尔

# f-string: Python 3.6+ 的现代字符串插值
# f-string: modern string interpolation (Python 3.6+)
print(f'{name} has {n_nodes:,} nodes')          # :,  千位分隔符
print(f'density = {density:.4%}')               # :.4%  百分号格式保留4位
print(f'directed = {is_directed!r}')            # !r  repr 格式

# type() 查看类型 / isinstance() 判断类型
print(type(n_nodes))                            # <class 'int'>
print(isinstance(n_nodes, int))                 # True

### 0.2 容器类型 Container Types

| 类型 | 字面量 | 可变 Mutable | 有序 Ordered | 元素唯一 Unique |
|---|---|:---:|:---:|:---:|
| `list` | `[1, 2, 3]` | ✅ | ✅ | ❌ |
| `tuple` | `(1, 2, 3)` | ❌ | ✅ | ❌ |
| `dict` | `{'a': 1}` | ✅ | ✅(3.7+) | key唯一 |
| `set` | `{1, 2, 3}` | ✅ | ❌ | ✅ |

In [ ]:
# list —— 有序、可重复、可修改
kinds = ['Compound', 'Gene', 'Disease', 'Gene']  # 允许重复
kinds.append('Anatomy')           # 末尾追加
print(kinds[0], kinds[-1])        # 正向/反向索引
print(kinds[1:3])                 # 切片 slice: index 1 到 2 (不含3)

# tuple —— 不可变，常用作 dict key
node_key = ('Compound', 'DB00945')   # (kind, identifier)
k, v = node_key                      # 拆包 unpacking
print(k, v)

# dict —— key-value 映射
node = {'kind': 'Compound', 'name': 'Aspirin', 'identifier': 'DB00945'}
print(node['name'])               # 直接访问
print(node.get('data', {}))       # .get(key, default) 安全取值，key不存在返回默认

# set —— 无序去重，成员检测 O(1)
core_kinds = {'Compound', 'Gene', 'Disease'}
print('Gene' in core_kinds)       # True  — O(1)
print('Anatomy' in core_kinds)    # False

### 0.3 循环与条件 Loops & Conditionals

Python 用**缩进（4空格）**表示代码块，没有花括号 `{}`。  
Python uses **indentation (4 spaces)** for code blocks instead of curly braces.

In [ ]:
nodes = [
    {'kind': 'Compound', 'name': 'Aspirin'},
    {'kind': 'Gene',     'name': 'BRCA1'},
    {'kind': 'Disease',  'name': 'Breast Cancer'},
]

# for 循环
for node in nodes:
    if node['kind'] == 'Disease':              # if / elif / else
        print(f"Disease found: {node['name']}")

# enumerate: 同时获取下标和值
for i, node in enumerate(nodes):
    print(f'[{i}] {node["kind"]}: {node["name"]}')

# 多重赋值 / tuple unpacking
src_id = ('Compound', 'DB00945')
kind, identifier = src_id
print(kind, identifier)

# while 循环
count = 0
while count < 3:
    print(f'count = {count}')
    count += 1

### 0.4 函数 Functions

用 `def` 定义，参数可设**默认值**，调用时可用**关键字参数**指定。  
Defined with `def`. Parameters can have default values; call with keyword arguments for clarity.

In [ ]:
# 基础函数定义
def describe_graph(name, n_nodes, n_edges, directed=True):  # directed 有默认值
    direction = 'directed' if directed else 'undirected'
    return f'{name}: {n_nodes:,} nodes, {n_edges:,} edges ({direction})'

print(describe_graph('Hetionet', 47031, 2250197))
print(describe_graph('Karate', 34, 78, directed=False))   # 关键字参数

# *args: 接受任意数量位置参数，函数内部为 tuple
def total(*counts):
    return sum(counts)

print(total(1552, 20945, 137))  # 22634

# **kwargs: 接受任意关键字参数，函数内部为 dict
def make_node(**attrs):
    return attrs

print(make_node(kind='Gene', name='TP53', identifier=7157))

# 解包传参: ** 把 dict 展开成关键字参数
extra = {'kind': 'Compound', 'name': 'Aspirin'}
print(make_node(**extra))   # 等同于 make_node(kind='Compound', name='Aspirin')

### 0.5 类 Classes（快速入门）

EDA 代码不自定义类，但 `networkx.MultiDiGraph`、`pd.DataFrame` 等都是类的实例。  
理解三要素：`__init__`（构造）、`self`（自身引用）、`obj.method()`（调用方法）。  

The EDA doesn't define custom classes, but `nx.MultiDiGraph`, `pd.DataFrame` etc. are class instances.  
Key concepts: `__init__` (constructor), `self` (instance reference), `obj.method()` (method call).

In [ ]:
class Node:
    """表示知识图谱中的一个节点 / Represents a node in the knowledge graph."""

    def __init__(self, kind: str, identifier, name: str):
        # self.xxx = xxx 创建实例属性
        self.kind       = kind
        self.identifier = identifier
        self.name       = name

    def __repr__(self) -> str:           # 控制 print() 的输出格式
        return f'Node({self.kind}/{self.identifier})'

    def label(self) -> str:
        return f'{self.kind}::{self.name}'

    @property
    def key(self) -> tuple:              # @property: 像属性一样调用，不需要 ()
        return (self.kind, self.identifier)


gene = Node('Gene', 1956, 'EGFR')
print(gene)            # Node(Gene/1956)    ← __repr__
print(gene.kind)       # Gene               ← 实例属性
print(gene.label())    # Gene::EGFR         ← 方法调用
print(gene.key)        # ('Gene', 1956)     ← property

### 🏋️ 练习 Exercises — Ch.0

**E0.1** 用 f-string 打印 `'Hetionet: 47,031 nodes | 2,250,197 edges | density=0.0020%'`  
（数字从变量读取，密度用 `:.4%` 格式）

**E0.2** 给定边列表 `edges = [('Compound','treats','Disease'), ('Gene','associates','Disease')]`，  
用 for + tuple unpacking 打印每条边：`Compound --[treats]--> Disease`

**E0.3** 写函数 `count_kind(nodes, kind)` 返回属于指定 kind 的节点数量。

In [ ]:
# ── 参考答案 Reference Answers ───────────────────────────────────

# E0.1
n_nodes, n_edges = 47031, 2250197
density = n_edges / (n_nodes * (n_nodes - 1))
print(f'Hetionet: {n_nodes:,} nodes | {n_edges:,} edges | density={density:.4%}')

# E0.2
edges = [('Compound','treats','Disease'), ('Gene','associates','Disease')]
for src, rel, tgt in edges:
    print(f'{src} --[{rel}]--> {tgt}')

# E0.3
def count_kind(nodes, kind):
    return sum(1 for n in nodes if n['kind'] == kind)

sample = [{'kind':'Compound'},{'kind':'Gene'},{'kind':'Compound'}]
print(count_kind(sample, 'Compound'))  # 2

---
# Ch.1 数据结构进阶
## Advanced Data Structures

本章覆盖 `utils.py` 解析 Hetionet JSON 时的核心 Python 模式。  
Covers the core Python patterns used in `utils.py` when parsing Hetionet JSON.

### 1.1 推导式 Comprehensions

**原理 Principle**：推导式是将「循环 + 条件过滤 + 收集」压缩为单行的语法糖。  
底层等价于 for 循环，但更简洁，且 CPython 内部对其有专门优化（速度略快）。  

Comprehensions compress a loop + filter + collect pattern into one line.  
They are syntactic sugar with minor internal optimization in CPython.

In [ ]:
# ── List comprehension ────────────────────────────────────────────
# 等价 for 循环:
# result = []
# for n in nodes:
#     if n['kind'] == 'Compound':
#         result.append(n['name'])

nodes = [
    {'kind':'Compound','name':'Aspirin'},
    {'kind':'Gene',    'name':'TP53'},
    {'kind':'Compound','name':'Ibuprofen'},
    {'kind':'Disease', 'name':'Cancer'},
]

compound_names = [n['name'] for n in nodes if n['kind'] == 'Compound']
#                 ^^^^^^^^^  表达式      ^^^^^^^^^^^^^^^^^^^^^^^^^^^ 过滤条件(可省)
print(compound_names)   # ['Aspirin', 'Ibuprofen']

# ── Dict comprehension ────────────────────────────────────────────
# EDA 实际用法: name_lookup = {(n['kind'], n['identifier']): n['name'] for n in hetnet['nodes']}
name_lookup = {(n['kind'], n['name']): n['name'].upper() for n in nodes}
print(name_lookup)

# ── Set comprehension ─────────────────────────────────────────────
all_kinds = {n['kind'] for n in nodes}   # 自动去重
print(all_kinds)   # {'Gene', 'Compound', 'Disease'} (顺序不定)

# ── 嵌套推导式 Nested comprehension ──────────────────────────────
# 展平二维列表
matrix = [[1,2],[3,4],[5,6]]
flat = [x for row in matrix for x in row]
print(flat)   # [1, 2, 3, 4, 5, 6]

### 1.2 Tuple 作为复合键 Tuple as Composite Key

**原理 Principle**：dict 的 key 必须是**可哈希（hashable）**的对象，即不可变对象。  
tuple 不可变 → 可哈希 → 可作为 key。  
list 可变 → 不可哈希 → 不可作为 key（会抛 `TypeError`）。  

Hetionet 中每个节点的唯一标识是 `(kind, identifier)` 这个二元组，  
把它作为 NetworkX 节点的 ID 和 dict 的 key，可以同时携带两种信息。  

Dict keys must be **hashable** (immutable). Tuples are immutable → hashable → valid as keys.  
In Hetionet, `(kind, identifier)` is the natural composite key for each node.

In [ ]:
# Hetionet 节点 key: (kind, identifier)
node_attrs = {
    ('Compound', 'DB00945'): {'name': 'Aspirin'},
    ('Gene',     1956):      {'name': 'EGFR'},
    ('Disease',  'DOID:9074'):{'name': 'Lupus'},
}

key = ('Gene', 1956)
print(node_attrs[key]['name'])  # EGFR

# 为什么 list 不能做 key?
try:
    bad_dict = {['Gene', 1956]: 'test'}   # ← 这会报错
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'list'

# 实际代码中的用法 (utils.py to_networkx):
# for e in hetnet['edges']:
#     src = tuple(e['source_id'])   # list → tuple (可哈希)
#     tgt = tuple(e['target_id'])
#     G.add_edge(src, tgt, ...)

### 1.3 `collections.defaultdict`

**原理 Principle**：普通 dict 访问不存在的 key → `KeyError`。  
`defaultdict(factory)` 在 key 缺失时自动调用工厂函数创建默认值，  
免去「先判断 key 是否存在，再创建/追加」的样板代码。  

**常见工厂函数**：`list`（分组）、`set`（去重分组）、`int`（计数）、`dict`（嵌套）。  

Regular dict raises `KeyError` for missing keys.  
`defaultdict(factory)` auto-creates a default value using the factory, eliminating boilerplate checks.

In [ ]:
from collections import defaultdict

# ── defaultdict(list): 分组 grouping ──────────────────────────────
edges = [
    {'kind':'treats',     'src':'Aspirin',   'tgt':'Headache'},
    {'kind':'treats',     'src':'Ibuprofen', 'tgt':'Fever'},
    {'kind':'binds',      'src':'Aspirin',   'tgt':'COX1'},
]

by_kind = defaultdict(list)       # key 缺失时自动初始化为 []
for e in edges:
    by_kind[e['kind']].append(e)  # 不存在时自动建空 list

print(dict(by_kind))  # {'treats': [...], 'binds': [...]}

# ── defaultdict(set): 去重分组 (03_compound_disease.ipynb 中的用法) ──
compound_treats = defaultdict(set)
compound_treats['Aspirin'].add('Headache')
compound_treats['Aspirin'].add('Fever')
compound_treats['Aspirin'].add('Headache')   # 重复添加，set 自动去重
print(dict(compound_treats))  # {'Aspirin': {'Headache', 'Fever'}}

# ── defaultdict(int): 计数 counting ──────────────────────────────
counter = defaultdict(int)
for e in edges:
    counter[e['kind']] += 1    # key 缺失时默认值为 0
print(dict(counter))  # {'treats': 2, 'binds': 1}

### 1.4 `collections.Counter`

**原理 Principle**：`Counter` 是专为**计数**设计的 `defaultdict(int)` 特化版，  
额外提供 `.most_common(n)` 返回频率最高的 n 项，支持加减法合并。  

EDA 中用于统计各种 edge type 的数量，快速找到最常见的 metaedge。  

`Counter` is a specialized `defaultdict(int)` for counting, with `.most_common()` and arithmetic.

In [ ]:
from collections import Counter

edge_kinds = ['treats','treats','binds','associates','treats','binds']

# 方法1: 直接从可迭代对象构造 (最常用)
cnt = Counter(edge_kinds)
print(cnt)                    # Counter({'treats': 3, 'binds': 2, 'associates': 1})
print(cnt.most_common(2))     # [('treats', 3), ('binds', 2)]
print(cnt['treats'])          # 3  — 像 dict 一样访问
print(cnt['unknown'])         # 0  — 不存在的 key 返回 0 (不报错)

# 方法2: 手动累加 (03_compound_disease.ipynb 中的实际写法)
# kind_counts = Counter(e['kind'] for e in cd_edges)

# Counter 支持加法合并
extra = Counter({'treats': 2, 'regulates': 5})
combined = cnt + extra
print(combined.most_common(3))  # [('regulates',5),('treats',5),('binds',2)]

### 1.5 生成器表达式 Generator Expressions

**原理 Principle**：列表推导式 `[...]` 立即求值并把所有结果放入内存。  
生成器表达式 `(...)` **惰性求值（lazy evaluation）**——每次迭代只产生一个值，不占额外内存。  

当处理 Hetionet 的 **220 万条边**时，如果只需要统计，生成器可以避免建立 220 万个对象的中间 list：  
```python
# ❌ 浪费内存: 先建 list，再传给 Counter
Counter([e['kind'] for e in hetnet['edges']])

# ✅ 节省内存: 生成器直接喂给 Counter
Counter(e['kind'] for e in hetnet['edges'])
```

List comprehension `[...]` evaluates eagerly. Generator expression `(...)` is lazy — yields one value at a time.

In [ ]:
import sys

N = 1_000_000
data = list(range(N))

list_comp = [x * 2 for x in data]       # 立即建立 100万元素的 list
gen_expr  = (x * 2 for x in data)       # 只是一个「待执行的计划」，几乎不占内存

print(f'list comprehension: {sys.getsizeof(list_comp):>12,} bytes')
print(f'generator expr:     {sys.getsizeof(gen_expr):>12,} bytes')

# 生成器只能遍历一次，用 for 或 sum/list 等消费
total = sum(gen_expr)   # 逐个消费，内存峰值极低
print(f'sum = {total:,}')

# 生成器被消费后无法重复使用
print(f'after exhaustion: {list(gen_expr)}')  # [] 空了

### 1.6 嵌套 defaultdict — `defaultdict(lambda: defaultdict(set))`

**原理 Principle**：`defaultdict` 的工厂函数可以是任意**可调用对象**，
包括 lambda 表达式。当需要「两层分组」时，可以用嵌套 defaultdict。  

`defaultdict(list)` 的工厂是 `list`（类本身就是可调用的）。  
需要嵌套时写 `defaultdict(lambda: defaultdict(set))`——
外层缺失 key 时自动创建一个新的 `defaultdict(set)` 作为 value。  

**EDA 中的实际用法（`03_compound_disease.ipynb` Step 8）**：  
```python
# edge_index[(src_kind, edge_kind, tgt_kind)][src_id] → set(tgt_ids)
edge_index = defaultdict(lambda: defaultdict(set))
```
三元组 `(src_kind, edge_kind, tgt_kind)` 是外层 key，
`src_id` 是内层 key，`set(tgt_ids)` 是最终值。
这样可以在 O(1) 内查找「所有 Compound 通过 binds 连到哪些 Gene」。  

The factory can be any callable, including a lambda. Nested defaultdicts
let you build a two-level index in one pass — used for metapath enumeration.

In [ ]:
# 普通两层嵌套 dict 的样板代码 (繁琐)
# d = {}
# if outer_key not in d:
#     d[outer_key] = {}
# if inner_key not in d[outer_key]:
#     d[outer_key][inner_key] = set()
# d[outer_key][inner_key].add(value)

# defaultdict(lambda: defaultdict(set)) 等价，但无需判断
from collections import defaultdict

edge_index = defaultdict(lambda: defaultdict(set))

# 模拟构建 Hetionet edge_index
sample_edges = [
    {'source_id': ('Compound','A'), 'target_id': ('Gene','G1'), 'kind': 'binds'},
    {'source_id': ('Compound','A'), 'target_id': ('Gene','G2'), 'kind': 'binds'},
    {'source_id': ('Compound','B'), 'target_id': ('Gene','G1'), 'kind': 'binds'},
    {'source_id': ('Gene','G1'),    'target_id': ('Disease','D1'),'kind': 'associates'},
]

for e in sample_edges:
    sk, si = e['source_id']
    tk, ti = e['target_id']
    k = e['kind']
    edge_index[(sk, k, tk)][si].add(ti)   # 正向
    edge_index[(tk, k, sk)][ti].add(si)   # 反向 (双向索引)

# 查询: 所有 Compound 通过 binds 连接到哪些 Gene
cbg_key = ('Compound', 'binds', 'Gene')
print('Compound A binds:', edge_index[cbg_key]['A'])  # {'G1', 'G2'}
print('Compound B binds:', edge_index[cbg_key]['B'])  # {'G1'}

# 两跳路径 Compound → Gene → Disease (CbG-GaD)
gad_key = ('Gene', 'associates', 'Disease')
compound = 'A'
two_hop_diseases = set()
for gene in edge_index[cbg_key][compound]:          # A → {G1, G2}
    two_hop_diseases |= edge_index[gad_key][gene]   # G1 → {D1}
print(f'Compound {compound} → Gene → Disease:', two_hop_diseases)  # {'D1'}

# lambda 作为工厂函数的注意事项:
# ❌ defaultdict(defaultdict(set))  — 报错: defaultdict(set) 不可调用
# ✅ defaultdict(lambda: defaultdict(set))  — lambda 每次被调用时返回新实例

### 🏋️ 练习 Exercises — Ch.1

**E1.1** 给定 `hetnet_nodes`（含 `kind`/`name` 字段的 list），  
用 dict comprehension 构造 `{name: kind}` 映射（一行代码）。

**E1.2** 用 `defaultdict(set)` 从 `treats_edges` 列表构造反向索引 `{disease: set_of_compounds}`。

**E1.3** 用**生成器表达式**（不建中间 list）对 100 万条边的 `kind` 字段统计 Counter。  
与先建 list 再统计的方式对比内存用量（用 `tracemalloc`）。

In [ ]:
# ── 参考答案 Reference Answers ───────────────────────────────────

# E1.1
hetnet_nodes = [{'kind':'Compound','name':'Aspirin'},
                {'kind':'Gene',    'name':'TP53'},
                {'kind':'Disease', 'name':'Cancer'}]
name_to_kind = {n['name']: n['kind'] for n in hetnet_nodes}
print('E1.1:', name_to_kind)

# E1.2
treats_edges = [{'src':'Aspirin','tgt':'Headache'},
                {'src':'Ibuprofen','tgt':'Fever'},
                {'src':'Aspirin','tgt':'Fever'}]
disease_index = defaultdict(set)
for e in treats_edges:
    disease_index[e['tgt']].add(e['src'])
print('E1.2:', dict(disease_index))

# E1.3
import tracemalloc, random
random.seed(42)
big_kinds = [random.choice(['treats','binds','associates']) for _ in range(1_000_000)]

tracemalloc.start()
cnt_list = Counter(big_kinds)             # via list (already built)
_, peak_list = tracemalloc.get_traced_memory(); tracemalloc.reset_peak()

cnt_gen = Counter(k for k in big_kinds)  # via generator
_, peak_gen = tracemalloc.get_traced_memory(); tracemalloc.stop()

print(f'E1.3 list peak:      {peak_list/1e6:.1f} MB')
print(f'E1.3 generator peak: {peak_gen/1e6:.1f} MB')
print('Results equal:', cnt_list == cnt_gen)

---
# Ch.2 文件读取与路径处理
## File I/O, Path Handling & Type Hints

对应 `utils.py` 中 `load_hetnet()` 的完整实现。  
Corresponds to the complete implementation of `load_hetnet()` in `utils.py`.

### 2.1 `pathlib.Path` — 面向对象路径

**原理 Principle**：传统做法用字符串拼接路径：`os.path.join(base, 'data', 'file.json')`，  
在 Windows（反斜杠 `\`）和 macOS/Linux（斜杠 `/`）之间容易出问题。  

`pathlib.Path` 把路径封装成对象，用 `/` 运算符拼接（跨平台），并提供丰富的属性和方法。  

Traditional string path concatenation breaks across platforms. `pathlib.Path` is cross-platform,  
uses the `/` operator for joining, and provides rich attributes like `.parent`, `.stem`, `.suffix`.

In [ ]:
from pathlib import Path

p = Path('/Users/chengruzhang/Desktop/gv_project_graph/eda/utils.py')

print(p.name)           # utils.py        — 完整文件名
print(p.stem)           # utils            — 无扩展名部分
print(p.suffix)         # .py              — 扩展名
print(p.parent)         # .../eda          — 上一级目录
print(p.parent.parent)  # .../gv_project_graph — 上两级

# / 运算符拼接路径 (跨平台)
data = p.parent.parent / 'hetionet-main' / 'hetnet' / 'json' / 'hetionet-v1.0.json.bz2'
print(data)
print(data.exists())    # True / False
print(data.suffix)      # .bz2

# utils.py 中的实际写法:
# DATA_PATH = Path(__file__).parent.parent / 'hetionet-main' / ... / 'hetionet-v1.0.json.bz2'
# __file__ 是当前 .py 文件自身的路径

### 2.2 `with` 语句与上下文管理器

**原理 Principle**：打开文件后必须关闭，否则造成资源泄漏。  
`with` 语句（上下文管理器）保证无论是否发生异常，退出 `with` 块时都会自动调用 `__exit__`（关闭文件）。  

The `with` statement (context manager) guarantees cleanup (`__exit__`) even if an exception occurs.  
For files, this means the file handle is always closed properly.

In [ ]:
# 不用 with: 必须手动关闭，异常时可能泄漏
# f = open('file.txt')
# data = f.read()
# f.close()   # 如果上面报错，这行不会执行

# 用 with: 自动关闭，安全
import tempfile, os
tmp = tempfile.mktemp(suffix='.txt')
with open(tmp, 'w') as f:
    f.write('hello hetionet')  # 写入
# 退出 with 块后 f 自动关闭

with open(tmp, 'r') as f:
    content = f.read()         # 读取
print(content)                 # hello hetionet
os.unlink(tmp)                 # 清理临时文件

### 2.3 `bz2` + `json` — 读取压缩 JSON

**原理 Principle**：`hetionet-v1.0.json.bz2` 是「先 JSON 序列化，再 bz2 压缩」的文件。  
- `bz2` 是一种无损压缩算法，压缩比约 4:1，适合文本数据
- `bz2.open(path, 'rt')` 以文本模式打开并**透明解压**，不需要先解压到磁盘
- `json.load(f)` 把 JSON 字符串流解析为 Python `dict`

`.json.bz2` = JSON serialized then bz2-compressed. `bz2.open(..., 'rt')` decompresses  
transparently in text mode. `json.load()` parses the JSON stream into a Python dict.

In [ ]:
import bz2, json
from pathlib import Path

DATA_PATH = (Path('/Users/chengruzhang/Desktop/gv_project_graph')
             / 'hetionet-main' / 'hetnet' / 'json' / 'hetionet-v1.0.json.bz2')

with bz2.open(DATA_PATH, 'rt', encoding='utf-8') as f:
    #          ^^^^^^^^^  透明解压    ^^^  文本模式 = 'read text'
    hetnet = json.load(f)   # JSON → Python dict

print(type(hetnet))              # <class 'dict'>
print(list(hetnet.keys()))       # ['graph', 'nodes', 'edges']
print(f'nodes: {len(hetnet["nodes"]):,}')  # 47,031
print(f'edges: {len(hetnet["edges"]):,}')  # 2,250,197

# 一个节点的结构
print('\nSample node:')
print(hetnet['nodes'][0])

# 一条边的结构
print('\nSample edge:')
print(hetnet['edges'][0])

### 2.4 类型注解 Type Hints (Python 3.5+)

**原理 Principle**：Python 是动态类型语言，但可以用类型注解让代码**意图更清晰**，  
并让 IDE（VS Code / Cursor）提供自动补全和错误提示。  

**重要**：类型注解在运行时**不强制执行**，只是文档和静态分析的提示，不影响运行结果。  

Type hints clarify intent and power IDE autocomplete. They are **not enforced at runtime**.

In [ ]:
from typing import Optional

# 函数签名中的类型注解
# param: Type = default  →  带默认值的带注解参数
# -> ReturnType          →  返回值类型

def load_hetnet(path: Path = DATA_PATH) -> dict:
    """Load Hetionet JSON.bz2 → raw Python dict."""
    with bz2.open(path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# 变量注解
node_count: int = 47031
name_lookup: dict[str, str] = {}

# Optional[X] = X | None (该参数可以是 X 类型或 None)
def get_name(node_id: tuple, lookup: Optional[dict] = None) -> str:
    if lookup is None:
        return str(node_id)
    return lookup.get(node_id, 'Unknown')

print(get_name(('Gene', 1956)))            # ('Gene', 1956)
print(get_name(('Gene', 1956), {'gene': 'EGFR'}))  # Unknown

### 🏋️ 练习 Exercises — Ch.2

**E2.1** 用 `pathlib.Path` 构造路径 `<项目根>/eda/utils.py`，打印其 `.parent`、`.stem`、`.suffix`。

**E2.2** 用 `bz2.open` 只读取 `hetnet['nodes']` 的**前 5 个**节点并打印各字段（不需要加载全部数据）。  
提示：`json.load` 会加载全部，可以加载后切片。

**E2.3** 写一个带完整类型注解的函数 `count_by_kind(nodes: list[dict]) -> dict[str, int]`，  
返回每种 kind 的节点数量，用 Counter 实现。

In [ ]:
# ── 参考答案 ─────────────────────────────────────────────────────

# E2.1
p = Path('/Users/chengruzhang/Desktop/gv_project_graph') / 'eda' / 'utils.py'
print('E2.1:', p.parent, '|', p.stem, '|', p.suffix)

# E2.2
with bz2.open(DATA_PATH, 'rt', encoding='utf-8') as f:
    data = json.load(f)
for node in data['nodes'][:5]:
    print(node)

# E2.3
def count_by_kind(nodes: list[dict]) -> dict[str, int]:
    return dict(Counter(n['kind'] for n in nodes))

print('E2.3:', count_by_kind(data['nodes']))

---
# Ch.3 Pandas 数据分析
## Pandas for Data Analysis

对应 `01_overview.ipynb` 中的节点/边类型统计与 Gini 系数计算。  
Corresponds to node/edge type statistics and Gini coefficient in `01_overview.ipynb`.

### 3.1 Series 与 DataFrame

**原理 Principle**：
- **Series**：带标签索引（index）的一维数组。类比 Excel 的一列。
- **DataFrame**：带行列双索引的二维表，每列是一个 Series，共享行 index。类比 Excel 整张表。

为什么用 Pandas 而不是原生 dict/list？  
→ 内置批量运算（向量化）、统计函数、排序、分组——无需手写循环。

Why Pandas over plain dict/list? Built-in vectorized ops, statistics, sorting, grouping — no loops needed.

In [ ]:
import pandas as pd
import numpy as np

# ── Series ────────────────────────────────────────────────────────
s = pd.Series([47031, 2250197, 11, 24],
              index=['nodes','edges','node_types','edge_types'],
              name='hetionet_stats')
print(s)
print(s['nodes'])   # 按标签访问
print(s[0])         # 按位置访问

# ── DataFrame from list of dicts ─────────────────────────────────
records = [
    {'kind':'Compound',            'count':1552},
    {'kind':'Gene',                'count':20945},
    {'kind':'Disease',             'count':137},
    {'kind':'Anatomy',             'count':402},
    {'kind':'Biological Process',  'count':3289},
]
df = pd.DataFrame(records)
print('\n', df)
print('\nColumns:', df.columns.tolist())
print('Shape:  ', df.shape)   # (5, 2)
print('\ndtypes:\n', df.dtypes)

### 3.2 常用统计操作

**`value_counts()`**：统计 Series 中每个唯一值的出现次数，自动按频率降序排列。  
**`sort_values()`**：按列值排序（ascending=True/False）。  
**`groupby(col).size()`**：按列分组后统计每组数量。  
**`reset_index()`**：把 index 变成普通列（常用于 value_counts 后整理）。  

`value_counts()` counts each unique value. `groupby().size()` is equivalent but more flexible.

In [ ]:
# 用真实 hetnet 数据
node_kinds = [n['kind'] for n in hetnet['nodes']]
edge_kinds = [e['kind'] for e in hetnet['edges']]

# value_counts: 统计频率并降序排列
kind_series = pd.Series(node_kinds)
counts = kind_series.value_counts()
print('Node type counts:')
print(counts)

# reset_index + rename: 整理成干净的 DataFrame
df_nodes = counts.reset_index()
df_nodes.columns = ['kind', 'count']
print('\nAs DataFrame:')
print(df_nodes)

# sort_values: 按 count 升序 (便于 barh 图由小到大)
df_sorted = df_nodes.sort_values('count', ascending=True)
print('\nSorted ascending:', df_sorted['kind'].tolist())

### 3.3 Gini 系数 Gini Coefficient

**原理 Principle**：Gini 系数源自经济学，衡量分布的不均等程度：
- **Gini = 0**：完全均等（每种类型节点数完全相同）
- **Gini = 1**：完全不均等（所有节点属于同一类型）

**公式推导**：  
$$G = \frac{\sum_{i=1}^{n} (2i - n - 1) x_i}{n \sum_{i=1}^{n} x_i}$$  
其中 $x_i$ 是升序排列后的第 $i$ 个值。

**为什么在 EDA 中用 Gini**？Hetionet 中 Gene 节点有 20,945 个，Disease 只有 137 个，  
Gini 系数能用一个数字量化这种极度不均衡——高 Gini 意味着模型需要处理严重的类别不平衡。

In Hetionet, Gene nodes (20,945) vastly outnumber Disease nodes (137). High Gini = severe class imbalance.

In [ ]:
def gini(values) -> float:
    """Gini coefficient for a 1D array of non-negative values."""
    arr = np.sort(np.array(values, dtype=float))  # 升序排列
    n   = len(arr)
    idx = np.arange(1, n + 1)             # [1, 2, ..., n]
    # 等价于: G = (2 * sum(i * x_i) / (n * sum(x_i))) - (n+1)/n
    return float((2 * (idx * arr).sum()) / (n * arr.sum()) - (n + 1) / n)


node_counts_list = list(counts.values)
g = gini(node_counts_list)
print(f'Node type Gini coefficient: {g:.3f}')
# 0 = 完全均等, 1 = 完全不均等 | 0=perfect equality, 1=total inequality

# 对比：均匀分布 vs 极度不均衡
uniform  = [100, 100, 100, 100, 100]
skewed   = [1, 1, 1, 1, 100000]
print(f'Uniform  Gini: {gini(uniform):.3f}')   # 接近 0
print(f'Skewed   Gini: {gini(skewed):.3f}')    # 接近 1

# top-2 节点类型的占比
total = sum(node_counts_list)
top2  = sorted(node_counts_list, reverse=True)[:2]
print(f'Top-2 share: {sum(top2)/total:.1%}')

### 3.4 边类型密度 Edge Density Analysis

**原理 Principle**：对于一种 metaedge（如 CtD, Compound-treats-Disease），  
"密度"定义为实际边数与理论最大边数之比：  

$$\text{density} = \frac{|E|}{|V_{src}| \times |V_{tgt}|}$$

这个指标反映两类节点之间关系的**稀疏程度**。  
CtD 密度 ≈ 0.36% 说明 99.6% 的 Compound-Disease 对是「未知」，这是链接预测的基础。

Edge density = actual edges / theoretical max. Low density → sparse → rich link prediction opportunity.

In [ ]:
# 统计各 metaedge 的 src/tgt 节点类型数量
kind_counts_map = Counter(n['kind'] for n in hetnet['nodes'])

# 边类型统计
edge_df = pd.DataFrame([
    {'kind': e['kind'],
     'src_kind': e['source_id'][0],
     'tgt_kind': e['target_id'][0]}
    for e in hetnet['edges']
])

summary = (edge_df.groupby(['kind','src_kind','tgt_kind'])
           .size().reset_index(name='edge_count'))

# 计算密度
summary['src_n'] = summary['src_kind'].map(kind_counts_map)
summary['tgt_n'] = summary['tgt_kind'].map(kind_counts_map)
summary['max_possible'] = summary['src_n'] * summary['tgt_n']
summary['density'] = summary['edge_count'] / summary['max_possible']

print(summary.sort_values('density', ascending=False).head(10)
      [['kind','edge_count','density']].to_string(index=False))

### 3.5 `Series.str.contains()` 与条件颜色列表

**原理 Principle**：`pd.Series.str` 是字符串方法的访问器（accessor），
把 Python `str` 的方法向量化到整列。  
`.str.contains(pattern)` 支持**正则表达式**，返回 bool Series，可直接用于行过滤。  

`01_overview.ipynb` 中用它筛选含 `Compound` 和 `Disease` 的 metaedge：  
```python
prediction_targets = density_df[density_df['metaedge'].str.contains('Compound.*Disease')]
```
正则 `Compound.*Disease` = 「包含 Compound，之后任意字符，再包含 Disease」。  

**条件颜色列表**：列表推导式结合条件表达式可以逐行为柱/点赋颜色——
这是 Matplotlib 中给特定子集高亮的标准做法。  

`Series.str.contains(regex)` vectorizes regex matching over a column,  
returning a boolean mask. Conditional color lists use list comprehensions to highlight subsets.

In [ ]:
import pandas as pd

# 模拟 density_df
density_df = pd.DataFrame({
    'metaedge': [
        'Compound-treats-Disease',
        'Compound-binds-Gene',
        'Gene-associates-Disease',
        'Compound-palliates-Disease',
        'Gene-interacts-Gene',
    ],
    'density': [0.0036, 0.021, 0.0089, 0.0015, 0.0042],
})

# ── str.contains: 正则过滤 ────────────────────────────────────────
mask = density_df['metaedge'].str.contains('Compound.*Disease')
#                               ^^^^^^^^^^^ 正则: Compound + 任意字符 + Disease
print('Bool mask:', mask.values)
prediction_targets = density_df[mask]
print('\nPrediction targets (Compound↔Disease metaedges):')
print(prediction_targets)

# str 访问器的其他常用方法:
print('\nstartswith:', density_df['metaedge'].str.startswith('Compound').values)
print('split example:', density_df['metaedge'].str.split('-').iloc[0])  # ['Compound','treats','Disease']

# ── 条件颜色列表 Conditional color list ─────────────────────────────
# EDA 实际代码:
# colors = ['crimson' if 'Compound' in m and 'Disease' in m else 'steelblue'
#           for m in density_df_sorted['metaedge']]

import matplotlib.pyplot as plt
colors = [
    'crimson' if ('Compound' in m and 'Disease' in m) else 'steelblue'
    for m in density_df['metaedge']
]
print('\nColors per row:', colors)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(density_df['metaedge'], density_df['density'], color=colors)
ax.set_xscale('log')
ax.set_title('Edge density (red = drug-repositioning targets)')
plt.tight_layout(); plt.show()

### 🏋️ 练习 Exercises — Ch.3

**E3.1** 用 `pd.Series(node_counts_list).describe()` 打印 Hetionet 节点类型计数的统计摘要。

**E3.2** 从 `hetnet['edges']` 中筛选出 `kind == 'treats'` 的边，  
用 `value_counts()` 统计 Compound 出现的频率（即每个 Compound 治疗了多少 Disease）。

**E3.3** 计算 CbG（Compound-binds-Gene）的密度，与 CtD 密度对比，哪个更稀疏？

In [ ]:
# ── 参考答案 ─────────────────────────────────────────────────────

# E3.1
print('E3.1:')
print(pd.Series(node_counts_list).describe())

# E3.2
print('\nE3.2: Compound treat frequency (top 10):')
treats = [e for e in hetnet['edges'] if e['kind'] == 'treats']
compound_freq = pd.Series(
    [e['source_id'][1] if e['source_id'][0]=='Compound' else e['target_id'][1]
     for e in treats]
).value_counts()
print(compound_freq.head(10))

# E3.3
n_c = kind_counts_map['Compound']
n_g = kind_counts_map['Gene']
n_d = kind_counts_map['Disease']
n_cbg = sum(1 for e in hetnet['edges'] if e['kind'] == 'binds')
n_ctd = sum(1 for e in hetnet['edges'] if e['kind'] == 'treats')
print(f'\nE3.3: CbG density = {n_cbg}/{n_c}×{n_g} = {n_cbg/(n_c*n_g):.4%}')
print(f'      CtD density = {n_ctd}/{n_c}×{n_d} = {n_ctd/(n_c*n_d):.4%}')

---
# Ch.4 Matplotlib / Seaborn 可视化
## Data Visualization

本章覆盖 EDA 三个 notebook 中所有图表的绘图语法。  
Covers all plotting patterns used across the three EDA notebooks.

### 4.1 Figure / Axes 解剖与字体配置

**原理 Principle**：
- **Figure**：整张画布，可包含多个子图（Axes）
- **Axes**：单个坐标系，绘图的实际载体

最佳实践：用 `fig, ax = plt.subplots()` **显式**创建，不要用全局 `plt.plot()` API——  
多子图时全局 API 行为不可预测。  

`Figure` = the canvas. `Axes` = one coordinate system.  
Best practice: always use `fig, ax = plt.subplots()` for explicit, predictable control.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── 中文字体配置 (macOS) ──────────────────────────────────────────
# 不配置会显示方框或警告
matplotlib.rcParams['font.family']      = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False  # 防止负号显示为方框
matplotlib.rcParams['figure.dpi']       = 100

# ── Figure / Axes 基本结构 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))   # figsize=(宽英寸, 高英寸)

x = [1, 2, 3, 4, 5]
y = [20945, 3289, 2833, 1552, 137]
labels = ['Gene', 'Bio.Process', 'Mol.Func', 'Compound', 'Disease']

ax.bar(x, y, color='steelblue', edgecolor='black', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('节点数 Node count')
ax.set_title('Hetionet 节点类型分布（部分）')

plt.tight_layout()   # 自动调整间距，防止标签被截断
plt.show()

### 4.2 多子图 Subplots

**原理 Principle**：`plt.subplots(nrows, ncols)` 一次性创建多个坐标系，  
返回 `(Figure, ndarray_of_Axes)`。  
用 `axes[i]`（1行）或 `axes[i, j]`（多行多列）访问每个子图。  

`plt.subplots(nrows, ncols)` returns a Figure and an ndarray of Axes.  
Access each subplot with `axes[i]` (1 row) or `axes[i, j]` (multiple rows).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))   # 1行2列

# 左图: 节点类型水平条形图
node_labels = ['Gene','Bio.Proc','Mol.Func','Symptom','Compound','Pathway','Anatomy','Disease']
node_vals   = [20945, 3289, 2833, 3169, 1552, 923, 402, 137]
sorted_pairs = sorted(zip(node_vals, node_labels))  # 按数量升序
sv, sl = zip(*sorted_pairs)

axes[0].barh(sl, sv, color='steelblue')
axes[0].set_xlabel('节点数 Count')
axes[0].set_title('节点类型分布 Node Type Distribution')

# 右图: 对数坐标条形图 (log-scale)
axes[1].barh(sl, sv, color='tomato')
axes[1].set_xscale('log')     # ← x 轴对数化
axes[1].set_xlabel('节点数 (log scale)')
axes[1].set_title('同上，对数坐标 Log Scale')

plt.tight_layout()
plt.show()

### 4.3 对数坐标轴 Log Scale

**原理 Principle**：当数据跨越多个数量级时（如度数从 1 到 10,000），  
线性坐标下大多数数据点被挤压到角落，规律不可见。  

对数坐标把**乘法关系**变成**加法关系**——等比数列在对数轴上是等距的。  
幂律分布 $P(k) \propto k^{-\alpha}$ 在 log-log 坐标下呈**直线**，斜率即 $-\alpha$。  

Log scale compresses large ranges. Power-law distributions appear as straight lines on log-log axes.

In [ ]:
np.random.seed(42)
# 模拟幂律度数序列 (真实网络中常见)
degrees = np.random.zipf(2.3, 3000)
bins_linear = np.arange(1, 50)
bins_log    = np.logspace(0, np.log10(degrees.max()), 25)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 线性坐标: 看不出尾部规律
h, e, _ = axes[0].hist(degrees, bins=30, color='gray', edgecolor='none')
axes[0].set_title('线性坐标 Linear — 尾部被压扁')
axes[0].set_xlabel('Degree')

# log-log 坐标: 幂律变直线
hist, edges = np.histogram(degrees, bins=bins_log)
centers = np.sqrt(edges[:-1] * edges[1:])  # 几何均值作为 bin 中心
mask = hist > 0
axes[1].scatter(centers[mask], hist[mask], s=25, color='tomato', alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Degree (log)')
axes[1].set_ylabel('Count (log)')
axes[1].set_title('Log-Log — 幂律在此呈直线 Power law = straight line')

plt.tight_layout()
plt.show()

### 4.4 矩阵热图 `imshow` 与 Seaborn `heatmap`

**原理 Principle**：
- `ax.imshow(A)`：将二维 NumPy 数组的每个元素映射为一个像素颜色。  
  适合**大型稀疏矩阵**（如 CtD 邻接矩阵 1552×137）。
- `sns.heatmap(df)`：Seaborn 封装，适合**小型矩阵**，可自动加数值标注。  

`imshow` maps each array element to a pixel color — fast for large matrices.  
`sns.heatmap` is better for small annotated matrices (adds colorbar, text labels, etc.).

In [ ]:
np.random.seed(0)
# 模拟 Compound-Disease 邻接矩阵 (CtD 密度约 0.36%)
A = (np.random.rand(40, 15) < 0.15).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# imshow: 适合大矩阵，像素级渲染
im = axes[0].imshow(A, aspect='auto', cmap='Reds', interpolation='nearest')
axes[0].set_xlabel('Disease index')
axes[0].set_ylabel('Compound index')
axes[0].set_title('imshow — 大矩阵 Large sparse matrix')
plt.colorbar(im, ax=axes[0], fraction=0.03)

# seaborn heatmap: 适合小矩阵 + 标注
small_df = pd.DataFrame(A[:8, :6],
                        index  =[f'C{i}' for i in range(8)],
                        columns=[f'D{i}' for i in range(6)])
sns.heatmap(small_df, annot=True, fmt='d', cmap='Reds',
            linewidths=0.5, ax=axes[1], cbar=True)
axes[1].set_title('seaborn heatmap — 小矩阵带标注')

plt.tight_layout()
plt.show()

### 4.5 `ax.annotate()` — 散点图文字标注

**原理 Principle**：`ax.annotate(text, xy)` 在数据坐标 `xy` 处添加文字标注。  
关键参数：  
- `xy`：被标注的数据点坐标（数据坐标系）  
- `xytext`：标注文字的偏移量（`textcoords='offset points'` 时单位为点）  
- `fontsize`：字号  
- `arrowprops`：箭头样式（可选）  

`03_compound_disease.ipynb` 用它在 metapath coverage-recall 散点图上标注每个点的名字：  
```python
for _, row in mp_df.iterrows():
    ax.annotate(row['metapath'], (row['coverage_%'], row['ctd_recall_%']),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
```

`ax.annotate(text, xy)` adds text labels to data points. `xytext` + `textcoords='offset points'`  
shifts the label a few pixels away to avoid overlapping with the marker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 模拟 metapath coverage vs recall 数据 (来自 03_compound_disease.ipynb)
mp_data = {
    'CbG-GaD': (18.2, 72.3),
    'CuG-GaD': (5.1,  45.1),
    'CdG-GaD': (3.8,  38.7),
    'CtD':     (0.36, 100.0),
    'CpD':     (0.18, 41.2),
}

fig, ax = plt.subplots(figsize=(8, 5))

for name, (cov, rec) in mp_data.items():
    ax.scatter(cov, rec, s=80, zorder=3)
    ax.annotate(
        name,                          # 标注文字
        xy=(cov, rec),                 # 数据点坐标
        xytext=(5, 3),                 # 偏移量 (单位: points)
        textcoords='offset points',    # 偏移量的坐标系
        fontsize=9,
    )

ax.set_xlabel('Coverage over all C-D pairs (%)')
ax.set_ylabel('Recall over known CtD positives (%)')
ax.set_title('Metapath: coverage vs. CtD recall')
ax.set_xscale('log')
plt.tight_layout(); plt.show()

# 带箭头的标注 (适合强调单个异常点)
fig, ax = plt.subplots(figsize=(5, 4))
xs = np.random.rand(20); ys = np.random.rand(20)
ax.scatter(xs, ys, s=40, alpha=0.6)
ax.annotate(
    'outlier',
    xy=(xs[0], ys[0]),
    xytext=(xs[0]+0.2, ys[0]+0.2),
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red', fontsize=10
)
ax.set_title('Annotate with arrow')
plt.tight_layout(); plt.show()

### 🏋️ 练习 Exercises — Ch.4

**E4.1** 用 `plt.subplots(2, 2)` 画 4 个子图，分别展示：条形图、水平条形图、饼图、折线图。

**E4.2** 对 Hetionet 的边类型 edge_counts，在 log-y 坐标下画水平条形图，  
比较和线性坐标下哪种能更清楚地展示小类别。

**E4.3** 随机生成一个 20×20 的相关矩阵（`np.corrcoef(np.random.randn(20,50))`），  
用 `sns.heatmap` 画出来，使用 `cmap='RdBu_r'` 和 `vmin=-1, vmax=1`。

In [ ]:
# E4.1
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
nk = node_labels[:6]; nv = node_vals[:6]
axes[0,0].bar(nk, nv, color='steelblue');      axes[0,0].set_title('Bar'); axes[0,0].tick_params(axis='x',rotation=40)
axes[0,1].barh(nk, nv, color='salmon');         axes[0,1].set_title('Horizontal Bar')
axes[1,0].pie(nv, labels=nk, autopct='%1.0f%%');axes[1,0].set_title('Pie')
axes[1,1].plot(nk, nv, 'o-', color='green');    axes[1,1].set_title('Line'); axes[1,1].tick_params(axis='x',rotation=40)
plt.tight_layout(); plt.show()

# E4.2
edge_counts_ser = pd.Series(edge_kinds).value_counts().head(15)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
edge_counts_ser.plot(kind='barh', ax=axes[0]); axes[0].set_title('Linear')
edge_counts_ser.plot(kind='barh', ax=axes[1]); axes[1].set_xscale('log'); axes[1].set_title('Log-X')
plt.tight_layout(); plt.show()

# E4.3
corr_mat = np.corrcoef(np.random.randn(20, 50))
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, ax=ax,
            square=True, linewidths=0.1)
ax.set_title('20×20 Correlation Matrix')
plt.tight_layout(); plt.show()

---
# Ch.5 NetworkX 图论
## Graph Theory with NetworkX

对应 `utils.py` 中 `to_networkx()`/`build_subgraph()`，以及 `02_structure.ipynb` 和 `03_compound_disease.ipynb`。  
Corresponds to `utils.py`, `02_structure.ipynb` and `03_compound_disease.ipynb`.

### 5.1 图论基础概念

**原理 Principle**：

| 概念 | 定义 | Hetionet 中的对应 |
|---|---|---|
| **节点 Node** | 图中的实体 | 化合物、基因、疾病... |
| **边 Edge** | 节点间的关系 | treats, binds, associates... |
| **有向图 Directed** | 边有方向 A→B ≠ B→A | 大多数 metaedge |
| **无向图 Undirected** | 边无方向 A—B | Gene-interacts-Gene |
| **多重图 MultiGraph** | 两节点间允许多条不同类型的边 | Compound 与 Gene 之间可有 binds + upregulates |
| **度 Degree** | 节点连接的边数 | Gene 的度极高（网络 hub）|
| **连通分量** | 相互可达的节点集合 | Hetionet 有一个包含绝大多数节点的巨型分量 |

Hetionet 用 **MultiDiGraph**（有向多重图）表示：  
有向是因为 treats 从 Compound 指向 Disease，而非双向；  
多重是因为同一对节点之间可能有不同类型的边（如一个 Compound 既 binds 又 upregulates 同一 Gene）。

### 5.2 图的创建与操作

In [ ]:
import networkx as nx

# 四种图类型
# nx.Graph()        — 无向图
# nx.DiGraph()      — 有向图
# nx.MultiGraph()   — 无向多重图
# nx.MultiDiGraph() — 有向多重图 ← Hetionet 使用这种

G = nx.MultiDiGraph()

# 添加节点 (带任意属性)
G.add_node(('Compound','DB00945'), kind='Compound', name='Aspirin')
G.add_node(('Gene', 1956),         kind='Gene',     name='EGFR')
G.add_node(('Disease','DOID:9074'),kind='Disease',  name='Lupus')

# 添加边 (带任意属性)
G.add_edge(('Compound','DB00945'), ('Disease','DOID:9074'),
           kind='treats', direction='>')
G.add_edge(('Compound','DB00945'), ('Gene',1956),
           kind='binds',  direction='>')

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')

# 遍历节点 (data=True 获取属性 dict)
for node, attrs in G.nodes(data=True):
    print(f'  {node} → kind={attrs["kind"]}, name={attrs["name"]}')

# 遍历边
print()
for src, tgt, attrs in G.edges(data=True):
    print(f'  {src[1]} --[{attrs["kind"]}]--> {tgt[1]}')

In [ ]:
# 度数查询
compound = ('Compound','DB00945')
print(f'Total degree:  {G.degree(compound)}')
print(f'Out-degree:    {G.out_degree(compound)}')   # 指向外部的边数
print(f'In-degree:     {G.in_degree(compound)}')    # 从外部指入的边数

# 获取节点属性
print('Node attrs:', G.nodes[compound])

# 获取出边的邻居
print('Successors:', list(G.successors(compound)))

### 5.3 `to_networkx()` 逐行解读

这是 `utils.py` 核心函数的完整注解版本。  
Full annotated walkthrough of the core `to_networkx()` function.

In [ ]:
def to_networkx(hetnet: dict, directed: bool = True) -> nx.MultiDiGraph:
    """Convert raw Hetionet dict → NetworkX MultiDiGraph."""

    # 根据 directed 参数选择图类型
    G = nx.MultiDiGraph() if directed else nx.MultiGraph()

    for n in hetnet['nodes']:
        G.add_node(
            (n['kind'], n['identifier']),   # ← tuple 作为唯一节点 ID
            kind=n['kind'],
            name=n['name'],
            **n.get('data', {})            # ← 解包附加属性 (分子量、DOID 等)
        )                                  #   .get(key, default) 防止 KeyError

    for e in hetnet['edges']:
        src = tuple(e['source_id'])        # ← list → tuple (list 不可哈希，不能作节点 ID)
        tgt = tuple(e['target_id'])
        G.add_edge(src, tgt,
                   kind=e['kind'],
                   direction=e['direction'],
                   **e.get('data', {}))

    return G

# 构建完整图 (大约需要 30-60 秒)
print('Building full NetworkX graph...')
G_full = to_networkx(hetnet)
print(f'Nodes: {G_full.number_of_nodes():,}')
print(f'Edges: {G_full.number_of_edges():,}')

### 5.4 诱导子图 Induced Subgraph

**原理 Principle**：从图 G 中选取**指定节点集合**，以及这些节点之间**原有的全部边**，  
形成的新图称为**诱导子图（induced subgraph）**。  

**为什么用子图**：NetworkX 的中心性算法复杂度高：
- Betweenness centrality: $O(V \times E)$ → 47k 节点 × 225万边 = 需要数小时
- Closeness centrality: $O(V^2)$ → 47k² ≈ 22亿次操作

提取 Compound + Gene + Disease（约 22,000 节点）后，计算量缩小约 80%，几分钟内完成。  

Induced subgraph = select nodes + keep all edges between them.  
Using a ~22k-node subgraph makes O(V×E) centrality algorithms finish in minutes instead of hours.

In [ ]:
def build_subgraph(G: nx.MultiDiGraph, kinds: list[str]) -> nx.MultiDiGraph:
    """Extract an induced subgraph containing only nodes of specified kinds."""
    kinds_set = set(kinds)                    # set → O(1) 成员检测 (比 list 快)
    nodes = [
        n for n, d in G.nodes(data=True)
        if d['kind'] in kinds_set             # 只保留指定 kind 的节点
    ]
    return G.subgraph(nodes).copy()           # .copy() → 独立副本，可修改
    # G.subgraph() 返回 view（轻量，节省内存）
    # .copy() 将 view 物化为完整副本（独立于原图）


CORE_KINDS = ['Compound', 'Gene', 'Disease']
G_core = build_subgraph(G_full, CORE_KINDS)

print(f'Full graph:     {G_full.number_of_nodes():,} nodes, {G_full.number_of_edges():,} edges')
print(f'Core subgraph:  {G_core.number_of_nodes():,} nodes, {G_core.number_of_edges():,} edges')

# 各 kind 的节点数
from collections import Counter
kind_dist = Counter(d['kind'] for _, d in G_core.nodes(data=True))
print('\nKind distribution in subgraph:')
for k, v in kind_dist.most_common():
    print(f'  {k:12s}: {v:,}')

### 5.5 度分布与中心性 Degree Distribution & Centrality

**中心性（Centrality）** 衡量节点在网络中的「重要性」，不同指标侧重不同维度：

| 指标 | 含义 | 复杂度 | EDA 参数 |
|---|---|---|---|
| **Degree** | 直接连接数 | O(V+E) | 全图 |
| **Betweenness** | 经过该节点的最短路径占比（信息瓶颈） | O(V×E) | k=300 近似采样 |
| **Closeness** | 到达其他节点平均距离的倒数（扩散速度） | O(V²) | 500 节点采样 |

**Betweenness 高** → 该节点是网络中信息传递的必经之路（如代谢通路枢纽基因）  
**Closeness 高** → 该节点能快速扩散到图中的任何位置  

EDA 中 k=300 是近似算法：随机采样 300 个节点作为起始点计算最短路径，而非从全部节点出发。

In [ ]:
# 用 karate club 演示 (34节点, 快速)
G_kc = nx.karate_club_graph()

deg_cent = nx.degree_centrality(G_kc)
btw_cent = nx.betweenness_centrality(G_kc, k=20, normalized=True)  # k=采样节点数
clo_cent = nx.closeness_centrality(G_kc)

cent_df = pd.DataFrame({'degree': deg_cent,
                        'betweenness': btw_cent,
                        'closeness': clo_cent})
cent_df.index.name = 'node'
print('Top-5 by betweenness:')
print(cent_df.sort_values('betweenness', ascending=False).head())

# 三种中心性的散点矩阵
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(cent_df['degree'], cent_df['betweenness'], alpha=0.7, s=50)
axes[0].set_xlabel('Degree centrality'); axes[0].set_ylabel('Betweenness')
axes[0].set_title('Degree vs Betweenness')

axes[1].scatter(cent_df['degree'], cent_df['closeness'], alpha=0.7, s=50, color='tomato')
axes[1].set_xlabel('Degree centrality'); axes[1].set_ylabel('Closeness')
axes[1].set_title('Degree vs Closeness')
plt.tight_layout(); plt.show()

### 5.6 二部图可视化 Bipartite Graph

**原理 Principle**：二部图（bipartite graph）的节点分为两个不相交集合 U 和 V，  
所有边只存在于 U 和 V 之间，同一集合内部没有边。  

Hetionet 的 CtD（Compound-treats-Disease）子图天然是二部图：  
- U = Compound 节点集合  
- V = Disease 节点集合  
- 边 = treats 关系  

这是**链接预测（link prediction）**的经典设置——预测哪些 Compound-Disease 对存在未发现的 treats 关系。

Bipartite: two disjoint node sets, edges only between sets. CtD is naturally bipartite and  
is the perfect setup for drug repurposing link prediction.

In [ ]:
B = nx.Graph()

compounds = ['Aspirin','Ibuprofen','Metformin','Lisinopril','Atorvastatin']
diseases  = ['Headache','Fever','Diabetes','Hypertension','Heart Disease']
edges_bi  = [('Aspirin','Headache'),('Aspirin','Fever'),
              ('Ibuprofen','Fever'),('Metformin','Diabetes'),
              ('Lisinopril','Hypertension'),('Atorvastatin','Heart Disease'),
              ('Aspirin','Hypertension'),('Lisinopril','Heart Disease')]

B.add_nodes_from(compounds, bipartite=0)   # bipartite=0/1 标记所属集合
B.add_nodes_from(diseases,  bipartite=1)
B.add_edges_from(edges_bi)

# 手动 bipartite 布局: 化合物在左 (x=0), 疾病在右 (x=1)
pos = {c: (0, -i*1.2) for i, c in enumerate(compounds)}
pos.update({d: (2, -i*1.2) for i, d in enumerate(diseases)})

fig, ax = plt.subplots(figsize=(10, 6))
nx.draw_networkx_nodes(B, pos, nodelist=compounds, node_color='steelblue',
                       node_size=700, ax=ax)
nx.draw_networkx_nodes(B, pos, nodelist=diseases,  node_color='tomato',
                       node_size=700, ax=ax)
nx.draw_networkx_edges(B, pos, alpha=0.5, width=1.5, ax=ax)
nx.draw_networkx_labels(B, pos, font_size=9, ax=ax)

# 图例
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='Compound'),
                   Patch(color='tomato',    label='Disease')], loc='upper center')
ax.axis('off')
ax.set_title('Compound ↔ Disease Bipartite Graph (CtD)')
plt.tight_layout(); plt.show()

### 5.7 连通分量 · 最短路径 · 聚类系数
## Connectivity · Shortest Paths · Clustering Coefficient

对应 `02_structure.ipynb` Step 5。  

**连通分量 Connected Components**：图中相互可达的最大节点子集。  
`nx.connected_components()` **只适用于无向图**——有向图需先调用 `.to_undirected()`，  
或使用 `nx.weakly_connected_components()`（忽略方向）。  

**最短路径 Shortest Path**：两节点间经过边数最少的路径。  
全图两两最短路径是 O(V×E)，EDA 中随机采样 1000 对近似分布。  

**聚类系数 Clustering Coefficient**：衡量「邻居们彼此也是邻居的程度」，  
即三角形闭合率。值接近 0 = 网格/树状结构；接近 1 = 高度集团化（cliques）。  

`nx.connected_components()` requires an undirected graph — convert with `.to_undirected()`.  
Shortest path and clustering together characterize the 'small-world' property.

In [ ]:
import networkx as nx
import numpy as np
from tqdm import tqdm

# ── tqdm: 进度条 ─────────────────────────────────────────────────
# tqdm(iterable) 包装任意可迭代对象，自动显示进度条
# 在 EDA 中用于显示最短路径采样的进度
for _ in tqdm(range(5), desc='示例进度条'):
    pass  # 模拟耗时操作

# ── 连通分量 Connected Components ───────────────────────────────
# G_core 是 MultiDiGraph (有向), 必须先转无向
G_undirected = G_core.to_undirected()
#              ^^^^^^^^^^^^^^^^^^^^^^ 生成一个无向副本，保留所有节点和边

components = list(nx.connected_components(G_undirected))
comp_sizes = sorted((len(c) for c in components), reverse=True)

print(f'连通分量总数 Total components : {len(components)}')
print(f'最大分量     Giant component  : {comp_sizes[0]:,} nodes '
      f'({comp_sizes[0]/G_undirected.number_of_nodes():.1%})')
print(f'前5大分量    Top-5 sizes      : {comp_sizes[:5]}')

# ── 最短路径采样 Shortest-path sampling ─────────────────────────
giant_nodes = list(max(components, key=len))
giant = G_undirected.subgraph(giant_nodes).copy()

# np.random.default_rng(seed): 现代 NumPy 随机数 API (替代旧式 np.random.seed)
# 优点: 每个 RNG 实例独立，不依赖全局状态，更安全
rng = np.random.default_rng(42)
N_SAMPLES = 300   # 演示用小一点
sample_idx = rng.choice(len(giant_nodes), size=(N_SAMPLES, 2), replace=True)

path_lengths = []
for i, j in tqdm(sample_idx, desc='Sampling paths', leave=False):
    if i == j:
        continue
    try:
        d = nx.shortest_path_length(giant, giant_nodes[i], giant_nodes[j])
        path_lengths.append(d)
    except nx.NetworkXNoPath:
        pass    # 极少数孤立节点对 → 跳过

path_lengths = np.array(path_lengths)
print(f'\n采样对数 Sampled pairs: {len(path_lengths)}')
print(f'平均最短路径 Mean shortest path: {path_lengths.mean():.2f}')
print(f'最大最短路径 Max shortest path:  {path_lengths.max()}')

# ── 平均聚类系数 Average Clustering Coefficient ──────────────────
# 计算量较大，对 giant component 可能需要数十秒
# 此处用 karate_club 演示
G_kc = nx.karate_club_graph()
avg_cc = nx.average_clustering(G_kc)
print(f'\nKarate club avg clustering: {avg_cc:.4f}')

# 小世界网络特征 Small-world check:
# 短平均路径长度 + 高聚类系数 → Small-world network
# Hetionet 满足这一特征

In [ ]:
import matplotlib.pyplot as plt

# 最短路径分布直方图
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(path_lengths,
        bins=range(int(path_lengths.min()), int(path_lengths.max()) + 2),
        color='teal', edgecolor='black', align='left')
ax.set_xlabel('Shortest path length')
ax.set_ylabel('Frequency')
ax.set_title(f'Sampled shortest path lengths '
             f'(n={len(path_lengths)}, mean={path_lengths.mean():.2f})')
plt.tight_layout(); plt.show()

### 5.8 双向 edge_index 与 Metapath 枚举
## Bidirectional Edge Index & Metapath Enumeration

对应 `03_compound_disease.ipynb` Step 8。  

**原理 Principle**：直接在 NetworkX 图上枚举 Compound → 中间节点 → Disease 的两跳路径，
对 220 万条边来说需要反复遍历图结构，非常慢。  

更好的做法是**预构建 edge_index**（嵌套 defaultdict），把边按类型组织成
`(src_kind, edge_kind, tgt_kind) → {src_id → set(tgt_ids)}` 的索引，  
然后用集合运算在纯 Python 中枚举所有长度为 2 的 metapath，速度快 10-100 倍。  

**Metapath**：异质图（heterogeeous graph）中跨越多种节点/边类型的路径模式，  
如 `CbG-GaD` = Compound→(binds)→Gene→(associates)→Disease。  

Building an edge_index (nested defaultdict) pre-organizes edges by type,  
making two-hop enumeration 10-100x faster than traversing NetworkX directly.

In [ ]:
from collections import defaultdict

# ── 构建双向 edge_index ────────────────────────────────────────────
# edge_index[(src_kind, edge_kind, tgt_kind)][src_id] = set(tgt_ids)
# 双向: 同时建正向和反向索引，支持任意方向查询

edge_index = defaultdict(lambda: defaultdict(set))

print('Building edge index...', end=' ')
for e in hetnet['edges']:
    sk, si = e['source_id']
    tk, ti = e['target_id']
    k = e['kind']
    edge_index[(sk, k, tk)][si].add(ti)   # 正向 forward
    edge_index[(tk, k, sk)][ti].add(si)   # 反向 backward
print(f'done. {len(edge_index)} edge type keys')

# ── 枚举所有 Compound → X → Disease 的长度为2 Metapath ────────────
compound_ids = {n['identifier'] for n in hetnet['nodes'] if n['kind'] == 'Compound'}
disease_ids  = {n['identifier'] for n in hetnet['nodes'] if n['kind'] == 'Disease'}

# 找到所有 Compound→X 的边类型
c_to_x_keys = [k for k in edge_index if k[0] == 'Compound']
# 找到所有 X→Disease 的边类型
x_to_d_keys = [k for k in edge_index if k[2] == 'Disease']

# 按中间节点类型 X 匹配
metapaths = []
for e1 in c_to_x_keys:
    for e2 in x_to_d_keys:
        if e1[2] == e2[0]:   # X 匹配: e1 的 tgt_kind == e2 的 src_kind
            metapaths.append((e1, e2))

print(f'\nLength-2 metapaths (Compound→X→Disease): {len(metapaths)}')
for e1, e2 in metapaths[:8]:
    name = f'{e1[0]}-{e1[1]}-{e1[2]}-{e2[1]}-{e2[2]}'
    print(f'  {name}')

In [ ]:
# ── 统计每条 Metapath 的覆盖率与 CtD 召回率 ──────────────────────
# CtD pairs (正样本)
compound_treats_local = defaultdict(set)
for e in hetnet['edges']:
    if e['kind'] == 'treats':
        sk, si = e['source_id']; tk, ti = e['target_id']
        c, d = (si, ti) if sk == 'Compound' else (ti, si)
        compound_treats_local[c].add(d)
ctd_pairs_set = {(c, d) for c, ds in compound_treats_local.items() for d in ds}
n_ctd = len(ctd_pairs_set)
max_pairs = len(compound_ids) * len(disease_ids)

mp_rows = []
for e1_key, e2_key in metapaths[:5]:   # 只算前5条 (完整版在03_compound_disease.ipynb)
    src2int = edge_index[e1_key]        # compound → set(intermediates)
    int2tgt = edge_index[e2_key]        # intermediate → set(diseases)
    reachable = set()
    for c in compound_ids:
        for mid in src2int.get(c, ()):
            for d in int2tgt.get(mid, ()):
                if d in disease_ids:
                    reachable.add((c, d))
    name = f'{e1_key[0][:3]}-{e1_key[1][:3]}-{e2_key[1][:3]}-{e2_key[2][:3]}'
    recall = len(reachable & ctd_pairs_set) / n_ctd if n_ctd else 0
    mp_rows.append({'metapath': name, 'n_pairs': len(reachable),
                    'coverage': len(reachable)/max_pairs,
                    'ctd_recall': recall})

import pandas as pd
mp_df = pd.DataFrame(mp_rows).sort_values('ctd_recall', ascending=False)
print(mp_df.to_string(index=False, float_format='{:.3f}'.format))

### 5.9 DWPC — 度加权路径计数
## Degree-Weighted Path Count

对应 `03_compound_disease.ipynb` Step 8 中 `dwpc_length2()` 函数。  

**问题 Problem**：原始路径计数（path count）对「经过高度 hub 节点（如泛素化基因 UBC）  
的路径」给分过高——UBC 连接几乎所有蛋白质，任何 Compound 通过它都能到达很多 Disease，  
但这不代表真正的生物学关联。  

**解决方案 Solution**：DWPC（Degree-Weighted Path Count）用阻尼指数 $\alpha$ 对每次  
经过高度节点的路径降权：

$$\text{DWPC}(c, d \mid P) = \sum_{\text{walks } w} \prod_{(u,v) \in w} 
\frac{1}{\deg_P(u)^\alpha \cdot \deg_P(v)^\alpha}$$

其中 $\alpha = 0.4$（Himmelstein et al. 2017 的实验值），  
$\deg_P(u)$ 是节点 $u$ 在该 metapath 边类型下的度数。  
  
**直觉 Intuition**：一个 hub 基因连接 500 个化合物，那么每条经过它的路径权重除以 $500^{0.4} \approx 14$，  
而一个低度基因（只连接 3 个化合物）的路径权重只除以 $3^{0.4} \approx 1.6$——  
后者的路径更具生物学特异性，得分更高。  

DWPC down-weights paths through high-degree hub nodes using a damping exponent α=0.4.  
This is the core feature used in the original Hetionet drug-repurposing model.

In [ ]:
import numpy as np
from collections import defaultdict

def dwpc_length2(e1_key, e2_key, compound_ids, disease_ids, alpha=0.4):
    """
    Length-2 DWPC: Compound →(e1)→ Intermediate →(e2)→ Disease
    Returns dict: {(compound_id, disease_id): dwpc_score}
    """
    src2int = edge_index[e1_key]   # compound_id → set(intermediate_ids)
    int2tgt = edge_index[e2_key]   # intermediate_id → set(disease_ids)

    # Degree of each node under its edge type
    # deg_e1_src[c] = 该化合物在 e1 中的出度
    deg_e1_src = {c: len(s) for c, s in src2int.items()}

    # deg_e1_int[x] = 中间节点 x 在 e1 中的入度 (反向边数)
    deg_e1_int = defaultdict(int)
    for c, ints in src2int.items():
        for x in ints:
            deg_e1_int[x] += 1

    # deg_e2_int[x] = 中间节点 x 在 e2 中的出度
    deg_e2_int = {x: len(s) for x, s in int2tgt.items()}

    # deg_e2_tgt[d] = 疾病节点 d 在 e2 中的入度
    deg_e2_tgt = defaultdict(int)
    for x, tgts in int2tgt.items():
        for d in tgts:
            deg_e2_tgt[d] += 1

    scores = defaultdict(float)
    for c in compound_ids:
        dc_e1 = deg_e1_src.get(c, 0)
        if dc_e1 == 0:
            continue
        for x in src2int.get(c, ()):
            dx_e1 = deg_e1_int.get(x, 1)
            dx_e2 = deg_e2_int.get(x, 0)
            if dx_e2 == 0:
                continue
            # 经过 (c→x) 这条边的权重
            w_cx = 1.0 / (dc_e1 ** alpha * dx_e1 ** alpha)
            for d in int2tgt.get(x, ()):
                if d not in disease_ids:
                    continue
                dd_e2 = deg_e2_tgt.get(d, 1)
                # 经过 (x→d) 这条边的权重
                w_xd = 1.0 / (dx_e2 ** alpha * dd_e2 ** alpha)
                scores[(c, d)] += w_cx * w_xd

    return dict(scores)


# 用最佳 metapath 计算 DWPC
if metapaths:
    e1k, e2k = metapaths[0]
    print(f'Computing DWPC for {e1k} → {e2k} ...')
    best_scores = dwpc_length2(e1k, e2k, compound_ids, disease_ids)
    print(f'Scored pairs: {len(best_scores):,}')

    # CtD pairs 的 DWPC 分数 vs 其他 pairs
    ctd_pairs_set = {(c, d) for c, ds in compound_treats_local.items() for d in ds}
    pos_scores = [best_scores.get(p, 0.0) for p in ctd_pairs_set]
    neg_scores = [v for p, v in best_scores.items() if p not in ctd_pairs_set and v > 0]

    print(f'CtD (正样本) mean DWPC : {np.mean(pos_scores):.6f}')
    print(f'Other (负样本) mean DWPC: {np.mean(neg_scores):.6f}')
    print(f'正负样本均值比 Ratio: {np.mean(pos_scores)/np.mean(neg_scores):.1f}×')

### 🏋️ 练习 Exercises — Ch.5

**E5.1** 创建一个 `DiGraph`（5个节点，至少6条有向边），打印每个节点的 in/out degree，  
找出 in-degree 最高（最多被指向）的节点。

**E5.2** 在 `G_core` 上分别计算 Compound/Gene/Disease 节点的平均度数，  
用 boxplot 展示三种类型的度数分布（`pd.DataFrame({'kind':..., 'degree':...})`）。

**E5.3** 从 `G_core` 中提取只含 Compound 和 Disease 的子图，  
验证这个子图是否满足二部图条件（同类型节点间无边）。

In [ ]:
# E5.1
DG = nx.DiGraph()
DG.add_edges_from([(1,2),(1,3),(2,3),(3,4),(4,5),(5,1),(2,5),(3,5)])
for n in sorted(DG.nodes()):
    print(f'Node {n}: in={DG.in_degree(n)}, out={DG.out_degree(n)}')
top_in = max(DG.nodes(), key=lambda n: DG.in_degree(n))
print(f'Highest in-degree: node {top_in} (in={DG.in_degree(top_in)})')

# E5.2
deg_records = []
for n, d in G_core.nodes(data=True):
    if d['kind'] in ['Compound','Gene','Disease']:
        deg_records.append({'kind': d['kind'], 'degree': G_core.degree(n)})
deg_df = pd.DataFrame(deg_records)

fig, ax = plt.subplots(figsize=(7,4))
deg_df.boxplot(column='degree', by='kind', ax=ax)
ax.set_yscale('log')
ax.set_title('Degree distribution by node kind')
plt.suptitle('')
plt.tight_layout(); plt.show()

# E5.3
G_cd = build_subgraph(G_core, ['Compound','Disease'])
# 二部图条件：同类型节点间无边
same_kind_edges = [(u,v) for u,v,_ in G_cd.edges(data=True)
                   if G_cd.nodes[u]['kind'] == G_cd.nodes[v]['kind']]
print(f'\nE5.3: same-kind edges in C+D subgraph: {len(same_kind_edges)}')
print('→ Is bipartite:', len(same_kind_edges) == 0)

---
# Ch.6 NumPy / SciPy 科学计算与幂律拟合
## Scientific Computing: NumPy, SciPy & Power Law Fitting

对应 `02_structure.ipynb` Step 4 的度分布幂律拟合。  
Corresponds to Step 4 power-law fitting in `02_structure.ipynb`.

### 6.1 NumPy 数组基础

**原理 Principle**：Python 原生 list 的数值运算是逐元素 Python 循环，速度慢。  
NumPy `ndarray` 在底层是**连续的 C 内存块**，运算完全向量化——比 Python 循环快 100~1000 倍。  

EDA 中用 NumPy 做：对数变换、等比分桶（`logspace`）、矩阵构造、布尔掩码。  

NumPy's `ndarray` is a C-backed contiguous memory block. Vectorized ops are 100-1000x faster  
than Python loops. Used in EDA for log transforms, log-spaced histograms, matrix ops.

In [ ]:
import numpy as np

a = np.array([1, 5, 2, 8, 3])
print('shape:', a.shape)         # (5,) — 1维，5个元素
print('dtype:', a.dtype)         # int64

# 向量化运算 — 无需 for 循环
print(a * 2)                     # [2 10  4 16  6]
print(np.log(a))                 # 自然对数 element-wise
print(np.log10(a))               # 以10为底的对数

# 常用数组创建
print(np.zeros(4))               # [0. 0. 0. 0.]
print(np.arange(1, 6))           # [1 2 3 4 5]
print(np.linspace(0, 1, 5))      # 等差: [0, 0.25, 0.5, 0.75, 1.0]
print(np.logspace(0, 3, 4))      # 等比(以10为底): [1, 10, 100, 1000]

# 布尔索引 (masking) — EDA 中大量使用
mask = a > 3
print('mask:', mask)             # [False  True False  True False]
print('a[mask]:', a[mask])       # [5 8]  — 只保留满足条件的元素

# np.histogram: 统计频率分布
bins = np.logspace(0, 2, 10)     # 10个等比 bin
hist, edges = np.histogram([1,2,5,8,15,22,50,88,100], bins=bins)
print('hist:', hist)

### 6.1b 现代随机数 API — `np.random.default_rng()`

**原理 Principle**：旧式 `np.random.seed(42)` 修改**全局状态**，  
在多线程或模块化代码中会造成难以复现的 bug。  

Python 3.x / NumPy 1.17+ 推荐的做法是**显式创建 Generator 实例**：  
```python
rng = np.random.default_rng(42)   # 独立实例，不影响全局
```
每个 `rng` 实例有自己的随机状态，不同实例之间互不干扰。  

`02_structure.ipynb` 中就是用 `rng.choice()` 采样最短路径测试对的：
```python
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(nodes_list), size=(N_SAMPLES, 2), replace=True)
```

Old `np.random.seed()` mutates global state — unsafe in multi-threaded code.  
`np.random.default_rng(seed)` creates an isolated Generator instance — the modern approach.

In [ ]:
import numpy as np

# ── 旧式 vs 新式 API 对比 ────────────────────────────────────────

# 旧式 (全局状态, 不推荐)
np.random.seed(42)
old_sample = np.random.randint(0, 100, size=5)
print('旧式 Old API:', old_sample)

# 新式 (实例化 Generator, 推荐)
rng = np.random.default_rng(42)
new_sample = rng.integers(0, 100, size=5)
print('新式 New API:', new_sample)

# 两个独立 Generator, 互不影响
rng_a = np.random.default_rng(1)
rng_b = np.random.default_rng(2)
print('rng_a:', rng_a.integers(0, 10, 4))
print('rng_b:', rng_b.integers(0, 10, 4))
# rng_a 再取不受 rng_b 影响
print('rng_a again:', rng_a.integers(0, 10, 4))

# EDA 中的实际用法: 采样 (N, 2) 的索引矩阵
nodes_demo = ['A','B','C','D','E']
rng = np.random.default_rng(42)
pairs = rng.choice(len(nodes_demo), size=(6, 2), replace=True)
print('\n采样 pairs:', pairs)
print('对应节点:', [(nodes_demo[i], nodes_demo[j]) for i, j in pairs])

### 6.2 幂律分布 Power Law Distribution

**原理 Principle**：幂律分布是自然界和人工网络中广泛存在的规律：
$$P(k) \propto k^{-\alpha}$$
其中 $k$ 是度数，$P(k)$ 是度为 $k$ 的概率，$\alpha$ 是幂律指数（网络中通常 $2 < \alpha < 3$）。

**直觉 Intuition**：  
- 大多数节点度数很低（只连接少数邻居）
- 极少数 **hub 节点**度数极高（连接数百或数千个邻居）
- 这类网络称为 **无标度网络（scale-free network）**

**为什么在 Hetionet 中重要**：  
- 少数基因 hub 连接几千个化合物、疾病 → 这些基因是核心药物靶点  
- 删除随机节点影响小；但删除 hub 节点（如 TP53）会大幅破坏网络连通性  
- 幂律特征告诉我们需要对「高度 hub」节点做特殊的 degree normalization（如 DWPC）

Power law networks (scale-free): most nodes have low degree, a few hubs have very high degree.  
In Hetionet, hub genes connect to thousands of compounds/diseases → key drug targets.

In [ ]:
np.random.seed(42)
# Zipf 分布 ≈ 离散幂律，参数 a 就是 α
degrees = np.random.zipf(a=2.5, size=5000)

print(f'Degree range: [{degrees.min()}, {degrees.max()}]')
print(f'Mean degree:  {degrees.mean():.1f}')
print(f'Median:       {np.median(degrees):.1f}')
print(f'Max degree:   {degrees.max()}  (hub node!)')

# 线性 vs log-log 对比
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 线性: hub 把所有低度节点压缩到左边
axes[0].hist(degrees[degrees < 30], bins=30, color='gray', edgecolor='none')
axes[0].set_title('线性坐标 — hub 使数据倾斜')
axes[0].set_xlabel('Degree')

# log-log: 幂律呈直线
bins = np.logspace(0, np.log10(degrees.max()), 25)
hist, edges = np.histogram(degrees, bins=bins)
centers = np.sqrt(edges[:-1] * edges[1:])   # 几何均值
mask = hist > 0
axes[1].scatter(centers[mask], hist[mask], s=25, color='tomato')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_title('Log-Log — 幂律呈直线 Power law = straight line')
axes[1].set_xlabel('Degree (log)'); axes[1].set_ylabel('Count (log)')

plt.tight_layout(); plt.show()

### 6.3 `scipy.stats.linregress` — Log-Log 线性回归拟合 α

**原理 Principle**：  
$$P(k) = C \cdot k^{-\alpha}$$
两边取对数（以10为底）：
$$\underbrace{\log_{10} P(k)}_{y} = \underbrace{\log_{10} C}_{b} + \underbrace{(-\alpha)}_{m} \cdot \underbrace{\log_{10} k}_{x}$$

这是一个标准的线性方程 $y = b + m \cdot x$，可以用**最小二乘线性回归**拟合。  
`scipy.stats.linregress(x, y)` 返回：斜率 $m$、截距 $b$、相关系数 $r$、p值、标准误。  
**幂律指数 $\alpha = -m$（斜率的负值）**。  

Taking log of both sides transforms the power law into a linear equation y = b + mx.  
Linear regression on log-log data gives slope m = -α.

In [ ]:
from scipy.stats import linregress

# 用之前模拟的 degrees 数据
bins = np.logspace(0, np.log10(degrees.max()), 25)
hist, edges = np.histogram(degrees, bins=bins)
centers = np.sqrt(edges[:-1] * edges[1:])  # bin 中心 (几何均值)
mask = hist > 0

log_k = np.log10(centers[mask])   # x: log10(degree)
log_p = np.log10(hist[mask])      # y: log10(count)

# linregress 返回 5 个值:
slope, intercept, r_value, p_value, std_err = linregress(log_k, log_p)

alpha = -slope   # 幂律指数 α
r2    = r_value ** 2

print(f'Fitted α = {alpha:.3f}  (input Zipf a = 2.5)')
print(f'R²       = {r2:.4f}  (1.0 = perfect straight line)')
print(f'p-value  = {p_value:.2e}  (< 0.05 → statistically significant)')

# 可视化拟合结果
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(log_k, log_p, s=30, color='steelblue', alpha=0.8, zorder=3, label='Data')

x_fit = np.linspace(log_k.min(), log_k.max(), 100)
y_fit = intercept + slope * x_fit
ax.plot(x_fit, y_fit, 'r--', lw=2, label=f'Fit: α={alpha:.2f}, R²={r2:.3f}')

ax.set_xlabel('log₁₀(degree k)')
ax.set_ylabel('log₁₀(count)')
ax.set_title('Power-law fit in log-log space')
ax.legend()
plt.tight_layout(); plt.show()

### 6.4 完整流水线：图 → 度分布 → 幂律拟合

把所有步骤封装成函数，对应 `02_structure.ipynb` Step 4 的完整逻辑。  
Puts all steps together — the full pipeline as used in `02_structure.ipynb` Step 4.

In [ ]:
def fit_power_law(G: nx.MultiDiGraph, kind: str = None,
                  n_bins: int = 25) -> dict:
    """
    从 NetworkX 图提取指定 kind 节点的度数，拟合幂律指数 α。
    Returns dict with alpha, r2, n, mean_degree, max_degree.
    """
    # 1. 提取度数序列
    deg_list = [
        d for n, d in G.degree()
        if (kind is None or G.nodes[n].get('kind') == kind) and d > 0
    ]
    if len(deg_list) < 5:
        return {'error': 'Not enough data points'}

    degrees_arr = np.array(deg_list)

    # 2. 对数等间隔分桶直方图
    bins = np.logspace(0, np.log10(degrees_arr.max()), n_bins)
    hist, edges = np.histogram(degrees_arr, bins=bins)
    centers = np.sqrt(edges[:-1] * edges[1:])

    # 3. log-log 线性回归
    mask = hist > 0
    slope, _, r, _, _ = linregress(np.log10(centers[mask]),
                                   np.log10(hist[mask]))

    return {'alpha':       -slope,
            'r2':          r ** 2,
            'n':           len(degrees_arr),
            'mean_degree': float(degrees_arr.mean()),
            'max_degree':  int(degrees_arr.max())}


# 对 G_core 的三种节点类型分别拟合
print('Power-law fit results by node kind:')
print(f'{"Kind":15s} {"α":>6s}  {"R²":>6s}  {"n":>8s}  {"mean_deg":>10s}  {"max_deg":>8s}')
print('-' * 60)
for k in ['Compound', 'Gene', 'Disease']:
    res = fit_power_law(G_core, kind=k)
    if 'error' not in res:
        print(f'{k:15s} {res["alpha"]:6.2f}  {res["r2"]:6.3f}  '
              f'{res["n"]:8,}  {res["mean_degree"]:10.1f}  {res["max_degree"]:8,}')

### 🏋️ 练习 Exercises — Ch.6

**E6.1** 对比等差 bin（`np.linspace`）和等比 bin（`np.logspace`）在 log-log 图上的平滑度差异：  
用 `np.random.zipf(2.2, 3000)` 生成数据，分别画出两种 bin 的散点图。

**E6.2** 用 `fit_power_law()` 对 `G_core` 中**所有节点**（不按 kind 过滤）拟合幂律指数，  
并绘制实际数据点 + 拟合直线。

**E6.3** （探究题）幂律指数 α 越大意味着什么？  
生成 `zipf(2.0)` 和 `zipf(3.5)` 各 5000 个数，在同一 log-log 图上画出它们的度分布，  
对比两者的 hub 节点频率（alpha 小 → hub 更多 vs alpha 大 → hub 更少）。

In [ ]:
# E6.1: 等差 vs 等比 bin
np.random.seed(0)
z_data = np.random.zipf(2.2, 3000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (label, bins_arr) in enumerate([
        ('等差 bins (linspace)', np.linspace(1, z_data.max(), 30)),
        ('等比 bins (logspace)', np.logspace(0, np.log10(z_data.max()), 30))]):
    h, e = np.histogram(z_data, bins=bins_arr)
    c = (e[:-1]+e[1:])/2; m = h>0
    axes[i].scatter(np.log10(c[m]), np.log10(h[m]), s=20, color='steelblue')
    axes[i].set_title(f'{label}\n(in log-log space)')
    axes[i].set_xlabel('log10(k)'); axes[i].set_ylabel('log10(count)')
plt.tight_layout(); plt.show()

# E6.2: 全图幂律拟合
res_all = fit_power_law(G_core)
print(f'E6.2: All nodes α={res_all["alpha"]:.2f}, R²={res_all["r2"]:.3f}')

# E6.3: α 大小的含义
np.random.seed(1)
z2  = np.random.zipf(2.0, 5000)
z35 = np.random.zipf(3.5, 5000)

fig, ax = plt.subplots(figsize=(8, 5))
for arr, color, label in [(z2, 'tomato', 'α=2.0 (more hubs)'),
                           (z35,'steelblue','α=3.5 (fewer hubs)')]:
    bins = np.logspace(0, np.log10(arr.max()), 25)
    h, e = np.histogram(arr, bins=bins)
    c = np.sqrt(e[:-1]*e[1:]); m = h > 0
    ax.scatter(np.log10(c[m]), np.log10(h[m]/h[m].sum()), s=20, color=color, alpha=0.7, label=label)

ax.set_xlabel('log₁₀(degree)'); ax.set_ylabel('log₁₀(P(k))')
ax.set_title('E6.3: 幂律指数 α 大小的含义\nLarger α → steeper drop → fewer hubs')
ax.legend(); plt.tight_layout(); plt.show()

---
# 总结 Summary

恭喜完成全部 7 章！以下是 EDA 代码与本教程章节的对照索引。  
Congratulations on completing all 7 chapters!

## EDA 代码 ↔ 教程章节 Cross-Reference

| EDA 文件 | 核心用法 | 本教程章节 |
|---|---|---|
| `utils.py` `load_hetnet()` | `pathlib`, `bz2`, `json`, type hints | **Ch.2** |
| `utils.py` `to_networkx()` | `MultiDiGraph`, tuple key, `**kwargs` | **Ch.1.2**, **Ch.5.3** |
| `utils.py` `build_subgraph()` | set comprehension, `G.subgraph()` | **Ch.1.1**, **Ch.5.4** |
| `01_overview.ipynb` | `pd.DataFrame`, `value_counts()`, Gini | **Ch.3** |
| `01_overview.ipynb` | `str.contains()` regex, 条件颜色列表 | **Ch.3.5** |
| `01_overview.ipynb` | `barh`, `imshow`, `sns.heatmap` | **Ch.4** |
| `02_structure.ipynb` | `np.logspace`, `linregress`, log-log | **Ch.6** |
| `02_structure.ipynb` | 连通分量、最短路径、聚类系数 | **Ch.5.6** |
| `02_structure.ipynb` | `np.random.default_rng`, `tqdm` | **Ch.5.6**, **Ch.6.1b** |
| `02_structure.ipynb` | `betweenness_centrality`, `closeness_centrality` | **Ch.5.5** |
| `03_compound_disease.ipynb` | `defaultdict(set)`, `Counter`, bipartite | **Ch.1.3**, **Ch.5.6** |
| `03_compound_disease.ipynb` | 嵌套 `defaultdict(lambda: defaultdict(set))` | **Ch.1.6** |
| `03_compound_disease.ipynb` | `ax.annotate()` 散点标注 | **Ch.4.5** |
| `03_compound_disease.ipynb` | 双向 `edge_index` + metapath 枚举 | **Ch.5.7** |
| `03_compound_disease.ipynb` | DWPC 度加权路径计数 | **Ch.5.8** |

## Open-World Assumption (OWA) & 负采样设计

**`03_compound_disease.ipynb` Step 9 的核心建模风险**：

Hetionet 中约 **75% 的 Compound** 没有任何 CtD 边，但其中很多在药理上是活跃的  
（它们有 CbG 边，即已知结合靶点），只是尚未在临床上被关联到某个 Disease。  

把这些「未知」对全部视为负样本 → **违反开放世界假设（Open-World Assumption, OWA）**：
> 「没有记录到的关系」≠「该关系不存在」

**三种负采样策略对比**（EDA 中验证）：

| 策略 | 描述 | 风险 |
|---|---|---|
| 随机采样 | 从所有未知对均匀采样 | 偏向低度节点，负样本太「容易」|
| 度偏置采样 | 权重 ∝ √(deg_c × deg_d) | 负样本度分布接近正样本，更合理 |
| 仅活跃化合物 | 只对有 CbG 边的化合物采样负例 | 最保守，AUPRC 最高但样本量小 |

**建模建议**：  
1. 评估指标用 **AUPRC**（而非 AUROC）——极度不平衡时 AUROC 虚高  
2. 负采样用**度偏置策略**，避免模型学到 degree popularity bias  
3. 用 DWPC 特征替代原始路径计数，消除 hub 偏差  

## 推荐后续学习 Next Steps

1. **图神经网络 GNN**：PyG (PyTorch Geometric) 或 DGL——基于 Hetionet 做链接预测实验
2. **DWPC 特征**：Himmelstein et al. (2017) 原论文的完整 metapath 特征工程
3. **统计检验**：`scipy.stats.kstest` 检验度分布是否严格符合幂律
4. **大图可视化**：`pyvis`（交互式 web）或 Gephi（桌面端）
5. **负采样策略**：degree-biased negative sampling 的完整实现见 `03_compound_disease.ipynb` Step 9
